# Custom model 

This custom model is created to understand what handcart should look like. the main objective of this model is to understand how does a cart look like to be use

In [12]:
!pip install roboflow ultralytics wandb python-dotenv --quiet

In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="31qjdnFxfQvTcGiaucig")
project = rf.workspace("kss").project("trolley-h0c3d")
version = project.version(5)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to trolley-5 in yolov8:: 100%|██████████| 1000/1000 [00:00<00:00, 1379.88it/s]


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\jessl\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
data_location = r"C:\Users\jessl\repos\yolo-handcart-detection\src\data\trolley-5"

In [17]:
from dotenv import load_dotenv
import os
import wandb

load_dotenv()

wandb.login()

wandb: Currently logged in as: jesslynzhang26 (jesslynzhang26-smu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os
from ultralytics import YOLO
import wandb

# 1. Start W&B run (optional but gives more control)
run = wandb.init(
    entity=os.getenv("WANDB_ENTITY"),
    project= os.getenv("WANDB_PROJECT_NAME"),
    config={
        "model": "yolov8s",
        "epochs": 10,
        "imgsz": 640,
        "batch": 16,
        "learning_rate": 0.01,
    }
)

# 2. Load model
model = YOLO("yolov8s.pt")

# 3. Train YOLO (this automatically logs to W&B too)
results = model.train(
    data=os.path.join(data_location, "data.yaml"),
    epochs=10,
    imgsz=640,
    batch=16,
    project=os.getenv("WANDB_PROJECT_NAME"), 
    name="yolo_initial_cart_detection",
    save=True,
    plots=True
)

# 4. Finish W&B run
run.finish()

# can cosider to tune the yolo parameter to do augmentation

# results = model.train(
#     data=data_location,
#     epochs=50,
#     imgsz=640,
#     batch=16,

#     hsv_h=0.015,
#     hsv_s=0.7,
#     hsv_v=0.4,

#     fliplr=0.5,
#     mosaic=1.0,
#     mixup=0.1
# )

New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.70  Python-3.13.14 torch-2.12.1+cpu CPU (12th Gen Intel Core i7-12700H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\jessl\repos\yolo-handcart-detection\src\data\trolley-5\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, mome

# Evaluate the trained model

In [ ]:
from ultralytics import YOLO
import os

# load best trained model
model = YOLO("runs/detect/yolo_initial_cart_detection/weights/best.pt")

# run validation
metrics = model.val(data=data_location + '\data.yaml')

print("\n===== EVALUATION METRICS =====")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# visualize predictions on validation set
results = model.predict(
    source=os.path.join(data_location, "/valid/images"),
    conf=0.25,
    save=True,
    save_txt=True  
)

In [ ]:
results = model.predict(
    source=os.path.join(data_location, "test/images"),
    conf=0.25,
    save=True
)

In [ ]:
result = results[0]

for box in result.boxes:
    cls = int(box.cls[0])
    conf = float(box.conf[0])
    xyxy = box.xyxy[0].tolist()

    print(f"class={cls}, conf={conf:.2f}, box={xyxy}")